# Phase-Connected CW Sampler — Adaptive Empirical Covariance + Newton Distance Snap

This notebook demonstrates the CW (continuous wave) sampler for phase-connected pulsar timing array analyses. It uses a simplified toy likelihood (white noise only, no red noise or GWB) to focus on the sampling method itself.

## The Problem
We sample 8 CW source parameters (sky location, inclination, chirp mass, frequency, strain, phase, polarisation) plus one distance per pulsar. Pulsar distances create a **multimodal** likelihood — the mode spacing $\Delta L$ is set by the GW wavelength and sky geometry. The Fisher information matrix (Hessian) at fixed distances dramatically underestimates the true posterior width for CW parameters (by 100–15,000×), because in reality the distances mode-hop to compensate for CW changes.

## The Solution
The sampler learns the proposal covariance from the chain itself for the main CW parameters, then snaps the pulsar distances to the nearest likelihood peak after every CW parameter update (frequency, strain, sky location, etc).

Concretely:
1. **Phase 1 (burn-in):** Use Fisher eigenmodes as initial proposals with aggressive per-mode scale adaptation.
2. **Phase 2 (mid burn-in onward):** Switch to eigenmodes of the **empirical chain covariance**, which captures the actual posterior geometry including distance mode-hopping.
3. **Newton distance snapping:** After each CW eigenmode or frequency-distance step, use gradient-based Newton iterations to shift all pulsar distances to their nearest likelihood peak.

| Move | Probability | Description |
|------|------------|-------------|
| **CW Eigen + Dist Newton** | 35% | Step along one empirical eigenmode, Newton-snap all distances |
| **Distance within-mode** | 10% | Small 1D Fisher-width refinement of one pulsar's distance |
| **Prior-snap big jump** | 30% | Draw distance from EM prior, snap to nearest peak |
| **Coherent freq-dist** | 10% | Joint frequency shift + rescale distances + Newton snap |
| **CW Joint** | 15% | Full 8D proposal from empirical covariance Cholesky factor |

## Setup and Imports

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["JAX_DISABLE_MMAP_CACHE"] = "1"
os.environ["XLA_FLAGS"] = "--xla_gpu_autotune_level=2"

import glob, time
import numpy as np
import matplotlib.pyplot as plt

import jax
jax.config.update('jax_enable_x64', True)
jax.clear_caches()
import jax.numpy as jnp

import discovery as ds
from enterprise_extensions import load_feathers
from discovery.deterministic import make_phase_connected_binary
from discovery import const as disco_const
from discovery.deterministic import fpcmu_fast

print('Imports OK')

## Load Pulsars and EM Distance Priors

Load pulsar data products and extract the electromagnetic distance priors (mean and uncertainty) for each pulsar. These priors constrain pulsar distances in the likelihood.

In [ ]:
# Load pulsars and their EM distance priors
feather_dir = "../data_products/"
Npulsars = 5

disco_psrs = [ds.Pulsar.read_feather(f) for f in sorted(glob.glob(feather_dir + "*.feather"))][:Npulsars]
for psr in disco_psrs:
    psr.toaerrs = np.full_like(psr.toas, 1e-6, dtype=np.float32)
print(f"Loaded {len(disco_psrs)} pulsars: {[p.name for p in disco_psrs]}")

psrs_ent = load_feathers.load_feathers_from_folder(feather_dir)
ent_by_name = {p.name: p for p in psrs_ent}

dist_mu = []
dist_sig = []
for psr in disco_psrs:
    ep = ent_by_name[psr.name]
    mu = float(ep.pdist[0])
    sig = float(ep.pdist[1]) if len(ep.pdist) > 1 else 0.5
    if (not np.isfinite(sig)) or sig <= 0:
        sig = 0.5
    dist_mu.append(mu)
    dist_sig.append(sig)

dist_mu = jnp.array(dist_mu, dtype=jnp.float64)
dist_sig = jnp.array(dist_sig, dtype=jnp.float64)

psr_toas_list = [np.asarray(psr.toas, dtype=np.float64) for psr in disco_psrs]
psr_pos_list = [psr.pos for psr in disco_psrs]
psr_positions = jnp.array([psr.pos for psr in disco_psrs])

sigma_toa = 1e-6
KPC_OVER_C = disco_const.kpc / disco_const.c

pnames_13 = (['cos_gwtheta', 'gwphi', 'cos_inc', 'log10_mc', 'log10_fgw',
              'log10_h', 'phase0', 'psi'] +
             [psr.name + '_dist' for psr in disco_psrs])

print(f"dist_mu: {[f'{float(d):.3f}' for d in dist_mu]}")
print(f"dist_sig: {[f'{float(d):.4f}' for d in dist_sig]}")

## CW Injection and Log-Posterior

Inject a single CW source into the data. The model uses `make_phase_connected_binary` from `discovery`, which computes the deterministic CW delay for each pulsar including both the Earth term and the pulsar term.

The log-posterior is:
$$\log p(\theta | d) = -\frac{1}{2} \sum_j \frac{(d_j - m_j(\theta))^2}{\sigma^2_{\rm TOA}} - \frac{1}{2} \sum_j \frac{(d_j^{\rm dist} - \mu_j^{\rm dist})^2}{\sigma_j^{{\rm dist}\,2}}$$

where $\theta$ = (8 CW params, $N_{\rm psr}$ distances), and the second term is the Gaussian EM distance prior.

In [ ]:
# CW injection and log-posterior definition
INJ = {
    "cos_gwtheta": 0.3, "gwphi": 2.5, "cos_inc": -0.2,
    "log10_mc": 9.0, "log10_fgw": -8.0, "log10_h": -12.0,
    "phase0": 1.0, "psi": 0.7,
}

cw_func = make_phase_connected_binary(pulsarterm=True)

@jax.jit
def compute_delta_L(cos_gwtheta, gwphi, log10_fgw):
    """Mode spacing in distance (kpc) for each pulsar.
    
    The pulsar-term CW likelihood is periodic in distance with spacing
    dL = 1 / (f_gw * (kpc/c) * |1 - cos(mu)|)
    where mu is the angle between the GW source and the pulsar.
    """
    gwtheta = jnp.arccos(cos_gwtheta)
    f_gw = 10.0 ** log10_fgw
    _, _, cos_mu = jax.vmap(
        lambda pos: fpcmu_fast(pos, gwtheta, gwphi)
    )(psr_positions)
    denom = jnp.abs(1.0 - cos_mu)
    denom = jnp.maximum(denom, 1e-4)
    return 1.0 / (f_gw * KPC_OVER_C * denom)

# Inject with distances offset from prior mean by 0.3 sigma
DIST_OFFSET_SIGMA = 0.3
p5_true_dists = {}
for i, psr in enumerate(disco_psrs):
    p5_true_dists[psr.name] = float(dist_mu[i]) + DIST_OFFSET_SIGMA * float(dist_sig[i])

p5_data_list = []
for i, psr in enumerate(disco_psrs):
    toas_i = np.asarray(psr.toas, dtype=np.float64)
    delay_i = cw_func(toas_i, psr.pos, p_dist=p5_true_dists[psr.name], **INJ)
    p5_data_list.append(np.array(delay_i, dtype=np.float64))

INJ_vals = [INJ[k] for k in ['cos_gwtheta','gwphi','cos_inc','log10_mc','log10_fgw','log10_h','phase0','psi']]
p5_dist_vals = [p5_true_dists[psr.name] for psr in disco_psrs]
p5_truth = np.array(INJ_vals + p5_dist_vals)

sd_arr = np.array([float(dist_sig[i]) for i in range(Npulsars)])
mu_arr = np.array([float(dist_mu[i]) for i in range(Npulsars)])

# Log-posterior
sd_jnp = jnp.array(sd_arr, dtype=jnp.float64)
data_jnp = [jnp.array(d) for d in p5_data_list]

@jax.jit
def logp(x):
    p_dists = x[8:13]
    in_bounds = (
        (x[0] >= -1) & (x[0] <= 1) &
        (x[1] >= 0) & (x[1] <= 2*jnp.pi) &
        (x[2] >= -1) & (x[2] <= 1) &
        (x[3] >= 7) & (x[3] <= 10) &
        (x[4] >= -9) & (x[4] <= -7) &
        (x[5] >= -18) & (x[5] <= -11) &
        (x[6] >= 0) & (x[6] <= 2*jnp.pi) &
        (x[7] >= 0) & (x[7] <= jnp.pi) &
        jnp.all(p_dists > 1e-6)
    )
    ll = 0.0
    for p_idx in range(5):
        model = cw_func(
            psr_toas_list[p_idx], psr_pos_list[p_idx],
            cos_gwtheta=x[0], gwphi=x[1], cos_inc=x[2],
            log10_mc=x[3], log10_fgw=x[4], log10_h=x[5],
            phase0=x[6], psi=x[7], p_dist=p_dists[p_idx], p_phase=None
        )
        resid = data_jnp[p_idx] - model
        ll -= 0.5 * jnp.sum(resid**2) / sigma_toa**2
    log_prior = -0.5 * jnp.sum(jnp.square((p_dists - dist_mu) / sd_jnp))
    return jnp.where(in_bounds, ll + log_prior, -1e30)

# Mode spacing at truth
dL_truth = np.array(compute_delta_L(INJ['cos_gwtheta'], INJ['gwphi'], INJ['log10_fgw']))

lp_truth = float(logp(jnp.array(p5_truth)))
print(f"logp(truth) = {lp_truth:.2f}")
for i, psr in enumerate(disco_psrs):
    print(f"  {psr.name}: d_true={p5_dist_vals[i]:.4f}, dL={dL_truth[i]:.6f}, "
          f"modes/sig={float(dist_sig[i])/dL_truth[i]:.0f}")

## Fisher Information and Proposal Setup

Compute the Hessian of the log-posterior at a reference point to set up the initial proposal distributions:
- **8×8 CW sub-block** → eigendecomposition gives eigenmode directions and widths for CW proposals
- **Distance diagonals** → per-pulsar within-mode step sizes and Newton snap curvatures

The Fisher eigenmodes are used during early burn-in. They will later be replaced by the empirical chain covariance.

In [ ]:
# JIT-compiled gradient for Newton snapping
grad_logp = jax.jit(jax.grad(logp))

def compute_fisher(x_peak, logp_fn):
    """Compute CW proposal components and distance Fisher widths at a mode peak.
    
    Returns dict with:
      - L_cw: 8x8 Cholesky for joint 8D proposal
      - eig_vecs: (8,8) eigenvectors of CW covariance (columns)
      - eig_sigs: (8,) per-eigenmode proposal widths (1D optimal scaling)
      - dist_fisher_sig: (Npsr,) per-pulsar within-mode widths
      - H_dist_diag: (Npsr,) Hessian diagonal for distance params (for Newton snap)
    """
    print("  Computing Hessian...")
    t0 = time.time()
    H_full = np.array(jax.hessian(logp_fn)(jnp.array(x_peak)))
    dt = time.time() - t0
    print(f"  Hessian computed in {dt:.1f}s")
    
    Npsr = len(x_peak) - 8
    
    # 8x8 CW sub-block (conditional on distances)
    H_cw = H_full[:8, :8]
    neg_H_cw = -H_cw
    eig_cw, evec_cw = np.linalg.eigh(neg_H_cw)
    eig_cw_c = np.maximum(eig_cw, 1e-12 * eig_cw.max())
    cov_cw = evec_cw @ np.diag(1.0 / eig_cw_c) @ evec_cw.T
    cov_cw = 0.5 * (cov_cw + cov_cw.T)
    
    # Joint 8D Cholesky
    scale_8d = 2.38**2 / 8
    L_cw = np.linalg.cholesky(scale_8d * cov_cw)
    
    # Eigenmode proposals
    eig_vals_cov, eig_vecs_cov = np.linalg.eigh(cov_cw)
    scale_1d = 2.38
    eig_sigs = scale_1d * np.sqrt(np.maximum(eig_vals_cov, 1e-30))
    
    print(f"\n  CW Fisher eigenvalues (precision): {eig_cw}")
    print(f"  Eigenmode proposal widths: {eig_sigs}")
    print(f"  Eigenvalue range: {eig_cw.max()/eig_cw.min():.1e}")
    
    # Per-pulsar distance Fisher widths + Hessian diagonals
    dist_fisher_sig = np.zeros(Npsr)
    H_dist_diag = np.zeros(Npsr)
    dL_at_peak = np.array(compute_delta_L(x_peak[0], x_peak[1], x_peak[4]))
    for j in range(Npsr):
        H_dist_diag[j] = H_full[8+j, 8+j]  # negative (concave)
        neg_hess_jj = -H_dist_diag[j]
        if neg_hess_jj > 0:
            dist_fisher_sig[j] = 1.0 / np.sqrt(neg_hess_jj)
        else:
            dist_fisher_sig[j] = dL_at_peak[j] * 0.3
    
    for j in range(Npsr):
        print(f"  Pulsar {j}: Fisher sig_d={dist_fisher_sig[j]:.6f}, "
              f"dL={dL_at_peak[j]:.6f}, ratio={dist_fisher_sig[j]/dL_at_peak[j]:.3f}")
    
    return dict(L_cw=L_cw, eig_vecs=eig_vecs_cov, eig_sigs=eig_sigs,
                dist_fisher_sig=dist_fisher_sig,
                H_dist_diag=H_dist_diag)

# Compute at truth
print("Fisher at truth:")
fisher = compute_fisher(p5_truth, logp)
print("Done")

## The Sampler

The sampler is a blocked Metropolis-Hastings MCMC with five move types, adaptive proposal scales, and a mid-burn-in switch from Fisher eigenmodes to empirical chain covariance.

Key components:
- **`snap_to_peak`**: Fine 1D scan to find the nearest distance mode peak for one pulsar
- **Newton distance snap**: After each CW eigenmode or freq-dist step, use 3 gradient-based Newton iterations to adjust all pulsar distances to their nearest peak. This is the key enabler — without it, CW proposals that change sky location or frequency would always land on a wrong distance mode and be rejected.
- **Empirical covariance switch**: At mid-burn-in, compute the sample covariance of CW parameters from the chain so far and use its eigenmodes instead of the Fisher matrix. This captures the true posterior geometry.

In [ ]:
def snap_to_peak(x13, pulsar_idx, dL_j, n_pts=50):
    """Find the nearest mode peak for one pulsar via fine 1D scan."""
    d_center = x13[8 + pulsar_idx]
    d_lo = max(d_center - 0.6 * dL_j, 1e-6)
    d_hi = d_center + 0.6 * dL_j
    d_candidates = np.linspace(d_lo, d_hi, n_pts)
    x_batch = np.tile(x13, (len(d_candidates), 1))
    x_batch[:, 8 + pulsar_idx] = d_candidates
    lps_scan = np.array(jax.vmap(logp)(jnp.array(x_batch)))
    return float(d_candidates[np.argmax(lps_scan)])


def ess_1d(x, max_lag=500):
    """Effective sample size from integrated autocorrelation time."""
    n = len(x)
    x = x - x.mean()
    var = np.var(x)
    if var < 1e-30:
        return 1.0
    acf = np.correlate(x, x, mode='full')[n-1:n-1+max_lag+1] / (n * var)
    cutoff = np.argmax(acf < 0.05)
    if cutoff == 0:
        cutoff = max_lag
    tau = 1.0 + 2.0 * np.sum(acf[1:cutoff])
    return n / max(tau, 1.0)


def run_sampler(logp_fn, x0, rng, fisher_dict, dL_arr, mu_arr, sig_arr,
                n_burn=3000, n_prod=10000, report_every=2000,
                p_cw_eigen=0.35, p_dist_within=0.10,
                p_big_jump=0.30, p_freq_dist=0.10,
                p_cw_joint=0.15,
                freq_dist_sigma=3e-4, n_newton=3,
                adapt_target=0.35, adapt_rate=0.8, adapt_t0=50,
                recompute_hessian_at=0.5,
                emp_cov_update_interval=5000,
                adapt_prod_steps=0,
                hessian_recompute_interval=3000):
    """Blocked MH sampler with adaptive empirical covariance + Newton snap.

    Phase 1 burn-in: Fisher eigenmodes with aggressive per-mode scale adaptation.
    Phase 2 (mid burn-in): switches to empirical CW covariance from accumulated
    burn-in samples, recomputes distance Hessian at MAP.  T=1 throughout.
    """
    D    = len(x0)
    Npsr = D - 8
    x  = x0.copy().astype(np.float64)
    lp = float(logp_fn(jnp.array(x)))

    L_cw            = fisher_dict['L_cw'].copy()
    eig_vecs        = fisher_dict['eig_vecs'].copy()
    eig_sigs        = fisher_dict['eig_sigs'].copy()
    dist_fisher_sig = fisher_dict['dist_fisher_sig'].copy()
    H_dist_diag     = fisher_dict['H_dist_diag'].copy()

    log_scale             = np.zeros(8)
    eig_adapt_count       = np.zeros(8)
    eig_window_acc        = np.zeros(8)
    eig_window_tot        = np.zeros(8)
    aggressive_check_interval = 50

    burn_samples_cw = []
    using_empirical  = False

    chain   = np.zeros((n_prod, D))
    lps_out = np.zeros(n_prod)

    acc = {'cw_eigen': 0, 'dist_within': 0, 'big_jump': 0,
           'freq_dist': 0, 'cw_joint': 0}
    tot = {k: 0 for k in acc}
    eig_acc = np.zeros(8)
    eig_tot = np.zeros(8)

    t1 = p_cw_eigen
    t2 = t1 + p_dist_within
    t3 = t2 + p_big_jump
    t4 = t3 + p_freq_dist

    total_steps = n_burn + n_prod
    switch_step = int(recompute_hessian_at * n_burn)
    t0 = time.time()

    for step in range(-n_burn, n_prod):
        abs_step = step + n_burn

        if step < 0:
            burn_samples_cw.append(x[:8].copy())

        # --- Switch to empirical covariance at mid burn-in ---
        if not using_empirical and abs_step == switch_step and len(burn_samples_cw) > 100:
            print(f"  [Switching to empirical CW covariance at burn-in step {abs_step}]")
            burn_cw = np.array(burn_samples_cw)
            emp_cov = np.cov(burn_cw.T)
            emp_cov = 0.5 * (emp_cov + emp_cov.T)
            ev, evec = np.linalg.eigh(emp_cov)
            ev = np.maximum(ev, 1e-30)

            cw_names_l = ['cos_gwtheta', 'gwphi', 'cos_inc', 'log10_mc',
                          'log10_fgw', 'log10_h', 'phase0', 'psi']
            print("  Empirical CW std vs Fisher eigenmode widths:")
            for ii, nn in enumerate(cw_names_l):
                print(f"    {nn:15s}: emp_std={np.sqrt(emp_cov[ii,ii]):.2e}, "
                      f"eig_sig={eig_sigs[ii]:.2e}")

            eig_vecs = evec
            eig_sigs = 2.38 * np.sqrt(ev)
            scale_8d = 2.38**2 / 8
            try:
                L_cw = np.linalg.cholesky(scale_8d * emp_cov)
            except np.linalg.LinAlgError:
                L_cw = np.linalg.cholesky(scale_8d * (emp_cov + 1e-10*np.eye(8)))

            print(f"  New eigenmode widths: {eig_sigs}")
            log_scale[:] = 0; eig_adapt_count[:] = 0
            eig_window_acc[:] = 0; eig_window_tot[:] = 0
            using_empirical = True

            x_snap = x.copy()
            for j in range(Npsr):
                dL_j = float(np.array(compute_delta_L(x[0], x[1], x[4]))[j])
                x_snap[8+j] = snap_to_peak(x_snap, j, dL_j)
            nf = compute_fisher(x_snap, logp_fn)
            dist_fisher_sig = nf['dist_fisher_sig']
            H_dist_diag     = nf['H_dist_diag']

        # --- Production empirical covariance update ---
        if using_empirical and step > 0 and step % emp_cov_update_interval == 0:
            recent = chain[max(0, step - emp_cov_update_interval):step, :8]
            if len(recent) > 100:
                ec = 0.5 * ((c := np.cov(recent.T)) + c.T)
                ev2, evec2 = np.linalg.eigh(ec)
                ev2 = np.maximum(ev2, 1e-30)
                eig_vecs = evec2
                eig_sigs = 2.38 * np.sqrt(ev2)
                try:
                    L_cw = np.linalg.cholesky((2.38**2 / 8) * ec)
                except np.linalg.LinAlgError:
                    pass

        r        = rng.random()
        do_adapt = (step < 0) or (step < adapt_prod_steps)

        if r < t1:
            # --- CW EIGENMODE + NEWTON DISTANCE SNAP ---
            # Step along one eigenmode of the (empirical or Fisher) CW covariance,
            # then Newton-snap all distances to their nearest peak.
            mode_idx   = rng.integers(8)
            z          = rng.standard_normal()
            scaled_sig = eig_sigs[mode_idx] * np.exp(log_scale[mode_idx])
            x_prop = x.copy()
            x_prop[:8] += z * scaled_sig * eig_vecs[:, mode_idx]

            # Newton snap: use gradient to adjust each distance toward its peak
            for _newton in range(n_newton):
                g = np.array(grad_logp(jnp.array(x_prop)))
                for j in range(Npsr):
                    if H_dist_diag[j] < -1e-6:
                        x_prop[8+j] = max(x_prop[8+j] - g[8+j] / H_dist_diag[j], 1e-6)

            lp_prop  = float(logp_fn(jnp.array(x_prop)))
            accepted = np.log(rng.random() + 1e-300) < lp_prop - lp
            if accepted:
                x = x_prop; lp = lp_prop
                acc['cw_eigen'] += 1; eig_acc[mode_idx] += 1
            tot['cw_eigen'] += 1; eig_tot[mode_idx] += 1

            # Adaptive scale: Robbins-Monro targeting adapt_target acceptance rate
            if do_adapt:
                eig_adapt_count[mode_idx] += 1
                gamma = adapt_rate / (eig_adapt_count[mode_idx] + adapt_t0)
                log_scale[mode_idx] += gamma * (float(accepted) - adapt_target)
                if step < 0:
                    eig_window_acc[mode_idx] += float(accepted)
                    eig_window_tot[mode_idx] += 1
                    if eig_window_tot[mode_idx] >= aggressive_check_interval:
                        wr = eig_window_acc[mode_idx] / eig_window_tot[mode_idx]
                        if wr > 0.90: log_scale[mode_idx] += np.log(2.0)
                        elif wr > 0.80: log_scale[mode_idx] += np.log(1.5)
                        eig_window_acc[mode_idx] = eig_window_tot[mode_idx] = 0
                log_scale[mode_idx] = np.clip(log_scale[mode_idx], -3.0, 10.0)

        elif r < t2:
            # --- DISTANCE WITHIN-MODE ---
            # Small Gaussian perturbation of one pulsar's distance using Fisher width.
            pi     = rng.integers(Npsr)
            x_prop = x.copy()
            x_prop[8+pi] += dist_fisher_sig[pi] * rng.standard_normal()
            if x_prop[8+pi] > 1e-6:
                lp_prop = float(logp_fn(jnp.array(x_prop)))
                if np.log(rng.random() + 1e-300) < lp_prop - lp:
                    x = x_prop; lp = lp_prop; acc['dist_within'] += 1
            tot['dist_within'] += 1

        elif r < t3:
            # --- PRIOR-SNAP BIG JUMP ---
            # Draw a distance from the EM prior, snap to the nearest likelihood peak.
            # This enables jumping between distance modes across the full prior range.
            pi     = rng.integers(Npsr)
            d_prop = rng.normal(mu_arr[pi], sig_arr[pi])
            if d_prop > float(dL_arr[pi]):
                x_snap = x.copy(); x_snap[8+pi] = d_prop
                d_snapped = snap_to_peak(x_snap, pi, float(dL_arr[pi]))
                x_prop = x.copy(); x_prop[8+pi] = d_snapped
                lp_prop = float(logp_fn(jnp.array(x_prop)))
                if np.log(rng.random() + 1e-300) < lp_prop - lp:
                    x = x_prop; lp = lp_prop; acc['big_jump'] += 1
            tot['big_jump'] += 1

        elif r < t4:
            # --- COHERENT FREQ-DIST ---
            # Jointly shift frequency and rescale all distances (since distance mode
            # spacing scales as 1/f_gw), then Newton-snap distances.
            delta  = rng.standard_normal() * freq_dist_sigma
            x_prop = x.copy(); x_prop[4] += delta
            if -9 <= x_prop[4] <= -7:
                sf = 10.0 ** (-delta)
                for j in range(Npsr): x_prop[8+j] *= sf
                for _newton in range(n_newton):
                    g = np.array(grad_logp(jnp.array(x_prop)))
                    for j in range(Npsr):
                        if H_dist_diag[j] < -1e-6:
                            x_prop[8+j] = max(x_prop[8+j] - g[8+j] / H_dist_diag[j], 1e-6)
                lp_prop = float(logp_fn(jnp.array(x_prop)))
                if np.log(rng.random() + 1e-300) < lp_prop - lp:
                    x = x_prop; lp = lp_prop; acc['freq_dist'] += 1
            tot['freq_dist'] += 1

        else:
            # --- CW JOINT 8D ---
            # Full 8-dimensional proposal from the (empirical or Fisher) Cholesky factor.
            z      = rng.standard_normal(8)
            x_prop = x.copy(); x_prop[:8] += L_cw @ z
            lp_prop = float(logp_fn(jnp.array(x_prop)))
            if np.log(rng.random() + 1e-300) < lp_prop - lp:
                x = x_prop; lp = lp_prop; acc['cw_joint'] += 1
            tot['cw_joint'] += 1

        if step >= 0:
            chain[step] = x; lps_out[step] = lp

        # Periodically recompute mode spacing (changes if CW params drift)
        if abs_step > 0 and abs_step % 2000 == 0:
            dL_arr = np.array(compute_delta_L(x[0], x[1], x[4]))

        # Periodic Hessian recomputation (proven to help when starting far from truth)
        if abs_step > 0 and abs_step % hessian_recompute_interval == 0 and using_empirical:
            nf2 = compute_fisher(x, logp_fn)
            dist_fisher_sig = nf2["dist_fisher_sig"]
            H_dist_diag = nf2["H_dist_diag"]
            # Update eigenmodes from recent chain
            if step >= 0 and step > 500:
                rc = chain[max(0, step-2000):step, :8]
                if len(rc) > 100:
                    ec = np.cov(rc.T); ec = 0.5*(ec + ec.T)
                    ev2, evec2 = np.linalg.eigh(ec)
                    ev2 = np.maximum(ev2, 1e-30)
                    eig_vecs = evec2
                    eig_sigs = 2.38 * np.sqrt(ev2)
                    try:
                        L_cw = np.linalg.cholesky((2.38**2 / 8) * ec)
                    except np.linalg.LinAlgError:
                        pass

        if abs_step > 0 and abs_step % report_every == 0:
            elapsed = time.time() - t0
            phase = 'burn' if step < 0 else 'prod'
            rate  = abs_step / elapsed
            eta   = (total_steps - abs_step) / rate
            ar_eig = acc['cw_eigen']  / max(tot['cw_eigen'],  1)
            ar_bj  = acc['big_jump']  / max(tot['big_jump'],  1)
            ar_fd  = acc['freq_dist'] / max(tot['freq_dist'], 1)
            ar_jt  = acc['cw_joint']  / max(tot['cw_joint'],  1)
            print(f"  [{phase} {abs_step}/{total_steps}] lp={lp:.2f} "
                  f"EIG={ar_eig:.3f} JT={ar_jt:.3f} BJ={ar_bj:.3f} FD={ar_fd:.3f} "
                  f"[{rate:.0f} it/s, ETA {eta:.0f}s]")

    dt_total = time.time() - t0
    ar_dict  = {k: acc[k] / max(tot[k], 1) for k in acc}
    print(f"\nDone in {dt_total:.1f}s ({total_steps/dt_total:.0f} it/s)")
    for k in acc:
        print(f"  {k:15s}: {acc[k]:6d}/{tot[k]:6d} = {ar_dict[k]:.4f}")
    print(f"\n  Per-eigenmode AR (adapted scale factors):")
    for i in range(8):
        ar_i = eig_acc[i] / max(eig_tot[i], 1)
        s_i  = np.exp(log_scale[i])
        print(f"    mode {i} (sig={eig_sigs[i]:.2e}, scale={s_i:.3f}, "
              f"eff_sig={eig_sigs[i]*s_i:.2e}): "
              f"{int(eig_acc[i])}/{int(eig_tot[i])} = {ar_i:.4f}")
    return chain, lps_out, ar_dict

print("run_sampler defined")

## Diagnostics

In [ ]:
def run_diagnostics(chain, lps, ar, truth, dL, title='', show_start=None):
    """Print diagnostics and plot traces / posteriors."""
    Npsr = chain.shape[1] - 8
    cw_keys = ['cos_gwtheta', 'gwphi', 'cos_inc', 'log10_mc',
               'log10_fgw', 'log10_h', 'phase0', 'psi']
    
    # Coverage
    cov = sum(1 for k in range(13)
              if np.percentile(chain[:,k], 5) <= truth[k] <= np.percentile(chain[:,k], 95))
    print(f"Coverage: {cov}/13")
    
    # ESS
    ess_vals = {pnames_13[k]: ess_1d(chain[:,k]) for k in range(13)}
    print(f"ESS range: [{min(ess_vals.values()):.0f}, {max(ess_vals.values()):.0f}]")
    for k, v in ess_vals.items():
        print(f"  {k:20s}: ESS={v:.0f}")
    
    # Distance errors
    print(f"\nDistance errors:")
    for j in range(Npsr):
        med = np.median(chain[:, 8+j])
        err_dL = abs(med - truth[8+j]) / dL[j]
        print(f"  {disco_psrs[j].name:13s}: median={med:.6f}, truth={truth[8+j]:.6f}, "
              f"err={err_dL:.1f} dL")
    
    # Acceptance rates
    print(f"\nAcceptance rates:")
    for k, v in ar.items():
        print(f"  {k:15s}: {v:.4f}")
    
    # --- Plots ---
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle(f'{title} (cov={cov}/13)', fontsize=14)
    
    # logp trace
    ax = axes[0, 0]
    ax.plot(lps, color='#1a3a5c', lw=0.4, alpha=0.8)
    ax.axhline(float(logp(jnp.array(truth))), color='r', ls='--', lw=1, label='truth')
    ax.set_xlabel('step'); ax.set_ylabel('logp')
    ax.set_title('Log-posterior trace'); ax.legend(fontsize=8)
    
    # CW param traces
    for i, idx in enumerate([0, 1, 4, 5]):
        ax = axes[0, 1] if i == 0 else axes[0, 2] if i == 1 else axes[1, 0] if i == 2 else axes[1, 1]
        ax.plot(chain[:, idx], color='#1a3a5c', lw=0.4, alpha=0.8)
        ax.axhline(truth[idx], color='r', ls='--', lw=1)
        ax.set_title(cw_keys[idx]); ax.set_xlabel('step')
    
    # Distance traces
    ax = axes[1, 2]
    colours = ['#1a3a5c', '#8b2500', '#2d5a27']
    for j in range(min(3, Npsr)):
        ax.plot(chain[:, 8+j], alpha=0.7, lw=0.4, color=colours[j],
                label=disco_psrs[j].name[:8])
        ax.axhline(truth[8+j], color=colours[j], ls='--', lw=0.8)
        if show_start is not None:
            ax.axhline(show_start[8+j], color=colours[j], ls=':', lw=0.8, alpha=0.5)
    ax.set_xlabel('step'); ax.set_ylabel('dist (kpc)')
    ax.set_title('Distance traces (first 3)'); ax.legend(fontsize=7)
    
    # Distance posteriors vs priors
    for j in range(min(3, Npsr)):
        ax = axes[2, j]
        ax.hist(chain[:, 8+j], bins=80, density=True, alpha=0.7, color='#2b5797')
        ax.axvline(truth[8+j], color='r', ls='--', lw=1.5, label='truth')
        ax.axvline(float(dist_mu[j]), color='green', ls=':', lw=1.5, label='prior mean')
        if show_start is not None:
            ax.axvline(show_start[8+j], color='orange', ls=':', lw=1.5, label='start')
        ax.set_title(f'{disco_psrs[j].name[:8]} dist'); ax.legend(fontsize=7)
    
    plt.tight_layout(); plt.show()
    return cov, ess_vals

print("Diagnostics function defined")

## Run the Sampler

Start from the injected truth values. With 5,000 burn-in steps and 50,000 production steps, the sampler should achieve ~60+ ESS for all CW parameters and 13/13 coverage.

In [ ]:
print("Starting sampler from truth")
print(f"  logp(truth) = {lp_truth:.2f}")

dL_start = np.array(compute_delta_L(p5_truth[0], p5_truth[1], p5_truth[4]))

chain, lps, ar = run_sampler(
    logp, p5_truth.copy(), np.random.default_rng(42),
    fisher_dict=fisher, dL_arr=dL_start, mu_arr=mu_arr, sig_arr=sd_arr,
    n_burn=5000, n_prod=50000, report_every=10000,
    adapt_prod_steps=0)

cov, ess = run_diagnostics(chain, lps, ar, p5_truth, dL_start,
                           title='Empirical Covariance + Newton Snap')